### Data info

https://gin.g-node.org/paolo_papale/TVSD

https://gin.g-node.org/paolo_papale/TVSD/issues/2

In the pre-processed data, the 1024 rows of e.g. "test_MUA" or "ALLMUA" to which ROI and array do belong?

The reply is different for the non-normalized MUA and for the normalized MUA.

for the normalized MUA (e.g. "test_MUA" variable):

This is simple, every 64 channels belong to an individual array, so 1:64 = array 1, 65:128 = array 2 etc. This is true for both monkeys.

The electrode to region association is the following:

In monkeyN
>>>
    1:512 =  V1
    513:768 = V4
    769:1024 = IT
In monkeyF
>>>
    1:512 =  V1
    513:832 = IT
    833:1024 = V4
for the non-normalized MUA (i.e. "ALLMUA" variable):

the association above is only correct after "mapping" the ALLMUA variable. What does mapping mean here? The recordings re-shuffle the electrodes because of how the headstages are designed. We use Blackrock Cereplex-M headtages, which cover 4 banks, but not in a sequential order (e.g. bank A covers channels 33:64, bank D covers channels 1:32).

So, if you want to use the non-normalized MUA, and you want to have the channels ordered according to the real arrangment of the arrays/ROIs, you need to do the mapping first!

In Matlab, that would simply be:

load([datadirgen,monkey,'\logs\1024chns_mapping_20220105.mat']);

ALLMUA = ALLMUA(mapping,:,:);

In [ ]:
from pathlib import Path
from collections import defaultdict
import json

import numpy as np
import mat73
from PIL import Image
from tqdm.auto import tqdm
import h5py


In [ ]:
from helpers import compute_ceiling_splithalf, compute_ceiling_variancebased

In [ ]:
channel2region = {
    "monkeyN": {
        "V1": (0, 511),
        "V4": (512, 767),
        "IT": (768, 1023),
    },
    "monkeyF": {
        "V1": (0, 511),
        "V4": (832, 1023),
        "IT": (512, 831),
    }
}

SUBJECTS = ['monkeyN', 'monkeyF']

ROIS = ['V1', 'V4', 'IT']

RELIABILITY = 0.1

In [ ]:
ds_path = '${MBS_THINGS_RAW_DIR}/TVSD'

ds_path = Path(ds_path)

list((ds_path / 'monkeyF').iterdir())

In [ ]:
neural_data = mat73.loadmat(ds_path / 'monkeyF' / 'THINGS_normMUA.mat')

In [ ]:
neural_data['test_MUA_reps'].shape

### Neural Data

In [ ]:
data_neural = defaultdict(dict)

for monkey in SUBJECTS:
    neural_data = mat73.loadmat(ds_path / monkey / 'THINGS_normMUA.mat')
    
    reliability = neural_data['reliab'].mean(axis=1)
    reliability = reliability >= RELIABILITY
    print(f"{monkey} reliability: {reliability.sum()}/{len(reliability)}")

    neural_responses_train = neural_data['train_MUA']
    neural_responses_test = neural_data['test_MUA_reps']

    for region, (start, end) in channel2region[monkey].items():
        neuroid_start, neuroid_end = start, end+1
        neural_responses_region_train = neural_responses_train[neuroid_start:neuroid_end].T
        neural_responses_region_train = np.expand_dims(neural_responses_region_train, -1)
        neural_responses_region_test = neural_responses_test[neuroid_start:neuroid_end]
        neural_responses_region_test = np.transpose(neural_responses_region_test, (1, 0, 2))

        neural_responses_region_train = neural_responses_region_train[:, reliability[neuroid_start:neuroid_end]]
        neural_responses_region_test = neural_responses_region_test[:, reliability[neuroid_start:neuroid_end]]
        
        data_neural[monkey][region] = {
            'train': neural_responses_region_train,
            'test': neural_responses_region_test
        }

        print(neural_responses_region_train.shape, neural_responses_region_test.shape)



In [ ]:
data_neural_train, data_neural_test = {}, {}
for monkey in SUBJECTS:
    data_neural_train[monkey] = {}
    data_neural_test[monkey] = {}
    for region in ROIS:
        data_neural_train[monkey][region] = np.array(data_neural[monkey][region]['train'])
        data_neural_test[monkey][region] = np.array(data_neural[monkey][region]['test'])

In [ ]:
for subject in SUBJECTS:
    for region in ROIS:
        print(subject, region, data_neural_train[subject][region].shape, data_neural_test[subject][region].shape)

In [ ]:
noise_ceiling_varbased = {}
noise_ceiling_splithalf = {}

for subject in tqdm(SUBJECTS):
    noise_ceiling_varbased[subject] = {}
    noise_ceiling_splithalf[subject] = {}
    for region in tqdm(ROIS, leave=False):
        data = data_neural_test[subject][region]
        data = np.transpose(data, (1, 0, 2))  # neuroids x images x repetitions
        noise_ceiling_varbased[subject][region] = compute_ceiling_variancebased(data)
        noise_ceiling_splithalf[subject][region] = compute_ceiling_splithalf(data).mean(-1)

In [ ]:
data_neural_train_avg = {
    sub: {
        roi: data_neural_train[sub][roi].mean(-1)
        for roi in ROIS
    }
    for sub in SUBJECTS
}

data_neural_test_avg = {
    sub: {
        roi: data_neural_test[sub][roi].mean(-1)
        for roi in ROIS
    } 
    for sub in SUBJECTS
}


In [ ]:
for sub in SUBJECTS:
    print("train", sub)
    for roi in ROIS:
        print(f"\t {roi} {data_neural_train_avg[sub][roi].shape}")
    
    print("\ntest", sub)
    for roi in ROIS:
        print(f"\t {roi} {data_neural_test_avg[sub][roi].shape}")
        print(f"\t {roi} nc {noise_ceiling_varbased[sub][roi].shape}")
    
    print("\n\n\n")

In [ ]:
# Checking the metadata

data1 = mat73.loadmat(ds_path / 'monkeyN' / '_logs' / 'things_imgs.mat')
data2 = mat73.loadmat(ds_path / 'monkeyF' / '_logs' / 'things_imgs.mat')


for split in ['train_imgs', 'test_imgs']:
    for key in ['class', 'things_path']:
        assert data1[split][key] == data2[split][key], f"Mismatch in {split} for key: {key}"


metadata = data1

In [ ]:
train_stimulus_ids = [img_path.replace('\\', '/') for img_path in metadata['train_imgs']['things_path']]

test_stimulus_ids = [img_path.replace('\\', '/') for img_path in metadata['test_imgs']['things_path']]

len(train_stimulus_ids), len(test_stimulus_ids)

### Concatenate data

In [ ]:
processed_data = {
    "train" :
        {
            "stimulus_ids": train_stimulus_ids,
            "neural_data": data_neural_train_avg,
        },
    "test" :
        {
            "stimulus_ids": test_stimulus_ids,
            "neural_data": data_neural_test_avg,
        },
    "noise_ceilings": noise_ceiling_varbased,
}

### Metadata

In [ ]:
metadata = {
    "reliability": RELIABILITY,
    "channel2region": channel2region,
    "desc": f"""
    The neural data is from the THINGS dataset, recorded from two monkeys (monkeyN and monkeyF).
    The neural data is recorded from V1, V4, and IT regions.
    The dataset includes both training and test splits.
    Channels with reliability less than {RELIABILITY} are excluded.
    """
}
metadata_str = json.dumps(metadata, indent=2)
metadata_str = json.dumps(metadata, indent=2).encode('utf-8')

### Save to disk

In [ ]:
data_dir = '${MBS_DATA_PREP_OUTPUT_DIR}'
filename = 'tvsd.h5'

data_dir = Path(data_dir)
data_path = data_dir / filename

In [ ]:
# processed_data[split]['labels']

In [ ]:
with h5py.File(data_path, 'w') as f:
    for split in ['train', 'test']:
        f.create_dataset(f"{split}/stimulus_ids", data=processed_data[split]['stimulus_ids'])

        for subj in tqdm(SUBJECTS):
            for roi in ROIS:
                f.create_dataset(f"{split}/neural_data/{subj}/{roi}", data=processed_data[split]['neural_data'][subj][roi])
                
    for subj in SUBJECTS:
        for roi in ROIS:
            f.create_dataset(f"noise_ceilings/{subj}/{roi}", data=processed_data['noise_ceilings'][subj][roi])

    f.attrs['metadata'] = metadata_str
    f.attrs['rois'] = ROIS
    f.attrs['subjects'] = list(SUBJECTS)
    f.attrs['splits'] = ['train', 'test']
    f.attrs['max_nc'] = 100
    f.close()


In [ ]:
loaded_data = defaultdict(dict)
with h5py.File(data_path, 'r') as f:
    splits = f.attrs['splits']
    subjects = f.attrs['subjects']
    rois = f.attrs['rois']
    for split in splits:
        loaded_data[split]['stimulus_ids'] = f[split]['stimulus_ids'][()]
        
        loaded_data[split]['neural_data'] = {}
        for subj in subjects:
            loaded_data[split]['neural_data'][subj] = {}
            for roi in rois:
                loaded_data[split]['neural_data'][subj][roi] = f[split]['neural_data'][subj][roi][()]
                
    for subj in subjects:
        loaded_data['noise_ceilings'][subj] = {}
        for roi in rois:
            loaded_data['noise_ceilings'][subj][roi] = f['noise_ceilings'][subj][roi][()]


In [ ]:
loaded_data['test'].keys()
loaded_data['test']['neural_data']['monkeyF']['IT'].shape, loaded_data['noise_ceilings']['monkeyF']['IT'].shape

In [ ]:

# Check the saved data
with h5py.File(data_path, 'r') as f:
    splits = f.attrs['splits']
    subjects = f.attrs['subjects']
    rois = f.attrs['rois']
    print(f.keys())
    for split in splits:
        print(f[split]['stimulus_ids'].shape)

        for subj in subjects:
            for region in rois:
                print(f[split]['neural_data'][subj][region].shape)
            
    print(json.loads(f.attrs['metadata']))